<a href="https://colab.research.google.com/github/iamatul1214/LLMs/blob/main/history_memory_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain Chat with History Memory

This notebook demonstrates how LangChain manages conversation history using `session_id`.
Sessions are stored in a plain Python dictionary — no DB required.

**Key idea:** When you ask *"What is the capital of India?"* and then *"How many states does it have?"*,
the LLM resolves *"it"* back to India because the full chat history is passed on every call.

## Step 1 — Install dependencies

In [1]:
# Run once
%pip install -q langchain langchain-community langchain-groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Step 2 — Imports

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Load .env from the parent directory (one level up from this notebook)
# load_dotenv(dotenv_path=Path(__file__).parent.parent / ".env" if "__file__" in dir() else Path("../.env"))

print("Imports OK")

Imports OK


## Step 3 — In-memory session store

A plain dict maps `session_id → ChatMessageHistory`.
`get_session_history` creates a new history on first access and returns the same one afterwards.

In [3]:
# ---- session store: just a dict, no DB ----
session_store: dict = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    """Return existing history or create a new one for this session_id."""
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
        print(f"  [store] Created new session: '{session_id}'")
    return session_store[session_id]

print("Session store ready")

Session store ready


## Step 4 — LLM + Prompt + Chain

We use **Groq** (cloud-hosted, free tier) running `llama-3.3-70b-versatile`.
The API key is loaded from the `.env` file in the project root.
The prompt has a `MessagesPlaceholder` called `history` — LangChain fills it automatically
before calling the LLM on every turn.

In [4]:
GROQ_API_KEY = ""  #Removed for security purposes
if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not found. Make sure it is set in your .env file as: groq_api_key = \"your_key\"")

In [5]:
GROQ_MODEL = "llama-3.3-70b-versatile"  # alternatives: "llama-3.1-8b-instant", "mixtral-8x7b-32768"

# ---- LLM (Groq cloud, open-source model) ----
llm = ChatGroq(model=GROQ_MODEL, temperature=0, api_key=GROQ_API_KEY)

# ---- Prompt template ----
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer concisely."),
    MessagesPlaceholder(variable_name="history"),  # full history injected here
    ("human", "{input}"),
])

# ---- Base chain: prompt -> LLM ----
chain = prompt | llm  # langchain's prompt-to-llm pipe operator (modern, idiomatic)

# ---- Wrap with automatic history management ----
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,           # factory function
    input_messages_key="input",
    history_messages_key="history",
)

print("Chain ready")
print(chain_with_history)

Chain ready
bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]) kwargs={} config={} config_factories=[] get_session_history=<function get_session_history at 0x79c3450100e0> input_messages_key='input' history_messages_key='history' history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)]


In [ ]:
chain

ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_c

## Step 5 — Helper utilities

In [6]:
def chat(session_id: str, user_message: str) -> str:
    """Send a message in a session and return the assistant reply."""
    config = {"configurable": {"session_id": session_id}}
    response = chain_with_history.invoke({"input": user_message}, config=config)
    return response.content


def show_history(session_id: str):
    """Pretty-print the stored message history for a session."""
    history = session_store.get(session_id)
    if not history:
        print(f"No history for session '{session_id}'")
        return
    print(f"\n--- History for session '{session_id}' ({len(history.messages)} messages) ---")
    for msg in history.messages:
        role = "Human" if msg.type == "human" else "AI   "
        print(f"  [{role}] {msg.content}")
    print("---\n")

print("Helpers ready")

Helpers ready


## Step 6 — Demo: contextual follow-up questions

Session `s1` asks about India, then uses *"it"* in the next question.
The LLM resolves the pronoun correctly because the history is always passed.

In [7]:
SESSION = "s1"

# Turn 1
q1 = "What is the capital of India?"
a1 = chat(SESSION, q1)
try:
    print(f"Q: {q1}")
except Exception as e:
    print("Error printing question:", e)
print(f"A: {a1}\n")

  [store] Created new session: 's1'
Q: What is the capital of India?
A: New Delhi.



In [8]:
# Turn 2 — 'it' refers to India; LLM knows because full history is in the prompt
q2 = "How many states does it have?"
a2 = chat(SESSION, q2)
print(f"Q: {q2}")
print(f"A: {a2}\n")

Q: How many states does it have?
A: 28 states and 8 union territories.



In [9]:
# Turn 3 — still about India
q3 = "What is the most populous state among them?"
a3 = chat(SESSION, q3)
print(f"Q: {q3}")
print(f"A: {a3}\n")

Q: What is the most populous state among them?
A: Uttar Pradesh.



In [10]:
# See what was stored
show_history(SESSION)


--- History for session 's1' (6 messages) ---
  [Human] What is the capital of India?
  [AI   ] New Delhi.
  [Human] How many states does it have?
  [AI   ] 28 states and 8 union territories.
  [Human] What is the most populous state among them?
  [AI   ] Uttar Pradesh.
---



## Step 7 — Multiple independent sessions

Session `s2` is completely isolated — it knows nothing about India.

In [11]:
SESSION2 = "s2"

r1 = chat(SESSION2, "What is the capital of France?")
print(f"Q: What is the capital of France?")
print(f"A: {r1}\n")

r2 = chat(SESSION2, "What language do they speak there?")
print(f"Q: What language do they speak there?")
print(f"A: {r2}\n")

show_history(SESSION2)

  [store] Created new session: 's2'
Q: What is the capital of France?
A: Paris.

Q: What language do they speak there?
A: French.


--- History for session 's2' (4 messages) ---
  [Human] What is the capital of France?
  [AI   ] Paris.
  [Human] What language do they speak there?
  [AI   ] French.
---



## Step 8 — Inspect the full session store

In [12]:
print("Active sessions:", list(session_store.keys()))
for sid, hist in session_store.items():
    print(f"  '{sid}': {len(hist.messages)} messages")

Active sessions: ['s1', 's2']
  's1': 6 messages
  's2': 4 messages


## How it works — summary

```
User input  (session_id = "s1")
      │
      ▼
RunnableWithMessageHistory
      │  1. get_session_history("s1")  →  looks up session_store["s1"]
      │  2. injects stored messages into {history} placeholder
      │  3. builds final prompt:  system + history + new human message
      │  4. calls LLM
      │  5. appends HumanMessage + AIMessage back into session_store["s1"]
      ▼
Assistant reply
```

Because the **full history is re-injected on every call**, the model can
resolve pronouns (*"it"*, *"they"*) and implicit follow-ups — no special NLP,
just context.